# P2 v2 — Reranker FT + QLoRA SFT
**Kaggle T4 x2 · CENG493 Turkish Legal RAG**

| Adım | Model | Süre (tahmini) |
|------|-------|----------------|
| 1 | Cross-Encoder Reranker FT | ~30 dakika |
| 2 | Qwen2.5-7B QLoRA SFT | ~90 dakika |
| 3 | Kaggle Dataset'e upload | ~5 dakika |

### v1'den Farklar
| | v1 (hatalı) | v2 (düzeltilmiş) | Neden |
|---|---|---|---|
| Reranker kısa query | Filtre yok | `< 3 kelime atlanır` | `"21. maddesi?"` gibi 1968 gürültülü örnek |
| Reranker class weight | Yok | `pos_weight=1.69` | Pos:Neg = 1:1.7 imbalance |
| LLM max_seq_length | **Yok** (OOM riski) | `1024` | %11 örnek 512'yi aşıyor, %0 1024'ü aşıyor |
| LLM max_grad_norm | `0.0` (kapalı) | `1.0` | Loss spike önlenir |
| LLM MAX_TRAIN | `2000` | `5000` | 13504 örnekten sadece 2000 kullanılıyordu |
| Upload yapısı | Ayrı ayrı (üst yazar) | Tek klasörden | S6/S7/S8 `MODEL_DIR/reranker_ft` + `MODEL_DIR/qwen_qlora_ft` bekliyor |

In [1]:
import os
os.environ['WANDB_DISABLED']          = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
print(f'GPU sayısı : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} | {p.total_memory/1024**3:.1f} GB')

GPU sayısı : 2
  GPU 0: Tesla T4 | 14.6 GB
  GPU 1: Tesla T4 | 14.6 GB


In [2]:
!pip install -q \
    sentence-transformers==3.4.1 \
    transformers \
    accelerate \
    bitsandbytes \
    peft \
    trl \
    datasets
print('Kurulum tamamlandı')

Kurulum tamamlandı


In [3]:
import json, random, gc, shutil
from dataclasses import dataclass
from pathlib import Path

@dataclass
class Config:
    # ── Veri yolları ─────────────────────────────────────────────
    DRIVE_DIR      : str = '/kaggle/input/datasets/ardayildiz29/legalo'
    RERANKER_FILE  : str = 'reranker.jsonl'
    LLM_SFT_FILE   : str = 'llm.jsonl'

    # ── Çıktı dizinleri ─────────────────────────────────────────
    RERANKER_OUT   : str = '/kaggle/working/reranker_ft'
    QLORA_OUT      : str = '/kaggle/working/qwen_qlora_ft'

    # ── Reranker ─────────────────────────────────────────────────
    RERANKER_BASE      : str   = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
    RERANKER_EPOCHS    : int   = 3
    RERANKER_LR        : float = 2e-5
    RERANKER_BATCH     : int   = 16      # cross-encoder küçük model, 16 güvenli
    RERANKER_MAX_LEN   : int   = 384
    RERANKER_MAX_TRAIN : int   = 9999    # 9999 = tüm filtrelenmiş veri
    RERANKER_MIN_QUERY : int   = 3       # bu kelimenin altındaki query atlanır

    # ── QLoRA ────────────────────────────────────────────────────
    LLM_MODEL       : str   = 'Qwen/Qwen2.5-7B-Instruct'
    QLORA_MAX_SEQ   : int   = 512     # %11 örnek 512'yi aşıyor, %0 1024'ü aşıyor
    QLORA_EPOCHS    : int   = 1
    QLORA_LR        : float = 2e-4
    QLORA_BATCH     : int   = 1
    QLORA_GRAD_ACC  : int   = 32
    QLORA_MAX_TRAIN : int   = 3000      # 13504 örnekten 5000 — ~90dk T4x2
    QLORA_R         : int   = 8
    QLORA_ALPHA     : int   = 16
    QLORA_DROPOUT   : float = 0.05

    # ── Kaggle Dataset ───────────────────────────────────────────
    KAGGLE_DATASET_ID : str = 'ardayildiz29/legal-models-tr-finetuned'

    RANDOM_SEED : int = 42

CFG = Config()
Path(CFG.RERANKER_OUT).mkdir(parents=True, exist_ok=True)
Path(CFG.QLORA_OUT).mkdir(parents=True, exist_ok=True)

for name, fname in [('reranker', CFG.RERANKER_FILE), ('llm', CFG.LLM_SFT_FILE)]:
    p = Path(CFG.DRIVE_DIR) / fname
    status = f'✅ OK ({p.stat().st_size/1024**2:.1f} MB)' if p.exists() else '❌ YOK'
    print(f'{name:<10}: {p}  {status}')

reranker  : /kaggle/input/datasets/ardayildiz29/legalo/reranker.jsonl  ✅ OK (7.2 MB)
llm       : /kaggle/input/datasets/ardayildiz29/legalo/llm.jsonl  ✅ OK (36.6 MB)


## Bölüm 1 — Cross-Encoder Reranker Fine-Tuning

In [4]:
from tqdm import tqdm

rk_path = Path(CFG.DRIVE_DIR) / CFG.RERANKER_FILE
rk_raw  = []
with open(rk_path, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            rk_raw.append(json.loads(line))

print(f'Ham reranker veri: {len(rk_raw)} satır')

# ── FİLTRELEME ─────────────────────────────────────────────────
# 1) Boş alan kontrolü
# 2) Kısa query filtresi: "21. maddesi?" gibi bağlamsız sorgular atlanır
random.seed(CFG.RANDOM_SEED)
rk_filtered = []
skipped_empty = 0
skipped_short = 0

for d in rk_raw:
    q   = d.get('query', '').strip()
    doc = d.get('candidate_passage', '').strip()
    lbl = d.get('label')
    if not (q and doc and lbl is not None):
        skipped_empty += 1
        continue
    if len(q.split()) < CFG.RERANKER_MIN_QUERY:
        skipped_short += 1
        continue
    rk_filtered.append(d)

pool = rk_filtered if len(rk_filtered) <= CFG.RERANKER_MAX_TRAIN else random.sample(rk_filtered, CFG.RERANKER_MAX_TRAIN)

pos = sum(1 for d in pool if int(d.get('label', 0)) == 1)
neg = len(pool) - pos
pos_weight = neg / max(pos, 1)

print(f'Atlanan (boş)  : {skipped_empty}')
print(f'Atlanan (kısa) : {skipped_short} (< {CFG.RERANKER_MIN_QUERY} kelime)')
print(f'Eğitim seti    : {len(pool)} örnek  |  pozitif: {pos}  negatif: {neg}')
print(f'pos_weight     : {pos_weight:.2f}  (class imbalance düzeltmesi)')

Ham reranker veri: 6357 satır
Atlanan (boş)  : 0
Atlanan (kısa) : 1968 (< 3 kelime)
Eğitim seti    : 4389 örnek  |  pozitif: 1634  negatif: 2755
pos_weight     : 1.69  (class imbalance düzeltmesi)


In [5]:
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

RK_OUT = Path(CFG.RERANKER_OUT)

if RK_OUT.exists() and (RK_OUT / 'config.json').exists():
    print(f'✅ Reranker zaten eğitilmiş: {RK_OUT}')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Reranker FT başlıyor | device: {device}')

    rk_tok   = AutoTokenizer.from_pretrained(CFG.RERANKER_BASE)
    rk_model = AutoModelForSequenceClassification.from_pretrained(
        CFG.RERANKER_BASE, num_labels=1
    ).to(device)

    class RerankerDataset(Dataset):
        def __init__(self, data):
            self.samples = [
                (d.get('query',''), d.get('candidate_passage',''), int(d.get('label',0)))
                for d in data if d.get('query') and d.get('candidate_passage')
            ]
        def __len__(self): return len(self.samples)
        def __getitem__(self, idx): return self.samples[idx]

    def collate_fn(batch):
        queries = [b[0] for b in batch]
        docs    = [b[1] for b in batch]
        labels  = torch.tensor([b[2] for b in batch], dtype=torch.float)
        enc = rk_tok(
            queries, docs,
            truncation=True, padding=True,
            max_length=CFG.RERANKER_MAX_LEN,
            return_tensors='pt'
        )
        return enc, labels

    rk_dataset = RerankerDataset(pool)
    rk_loader  = DataLoader(
        rk_dataset, batch_size=CFG.RERANKER_BATCH,
        shuffle=True, collate_fn=collate_fn, drop_last=True
    )

    total_steps  = len(rk_loader) * CFG.RERANKER_EPOCHS
    warmup_steps = int(total_steps * 0.1)
    optimizer    = AdamW(rk_model.parameters(), lr=CFG.RERANKER_LR)
    scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    # ── FİX: pos_weight ile class imbalance düzeltmesi ──────────
    pw       = torch.tensor([pos_weight], dtype=torch.float).to(device)
    loss_fn  = torch.nn.BCEWithLogitsLoss(pos_weight=pw)

    print(f'Toplam adım: {total_steps} | warmup: {warmup_steps}')
    print(f'Loss: BCEWithLogitsLoss(pos_weight={pos_weight:.2f})')

    rk_model.train()
    for epoch in range(CFG.RERANKER_EPOCHS):
        total_loss = 0.0
        for step, (enc, labels) in enumerate(tqdm(rk_loader, desc=f'Epoch {epoch+1}/{CFG.RERANKER_EPOCHS}')):
            enc    = {k: v.to(device) for k, v in enc.items()}
            labels = labels.to(device)
            outputs = rk_model(**enc)
            loss    = loss_fn(outputs.logits.squeeze(-1), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(rk_model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            total_loss += loss.item()
            if (step + 1) % 100 == 0:
                print(f'  step {step+1}/{len(rk_loader)}  avg_loss: {total_loss/(step+1):.4f}')
        print(f'Epoch {epoch+1} bitti | avg_loss: {total_loss/len(rk_loader):.4f}')

    rk_model.save_pretrained(str(RK_OUT))
    rk_tok.save_pretrained(str(RK_OUT))

    del rk_model, rk_tok, rk_loader, rk_dataset
    gc.collect()
    torch.cuda.empty_cache()

    print(f'\n✅ Reranker kaydedildi: {RK_OUT}')
    for f in sorted(RK_OUT.rglob('*')):
        if f.is_file():
            print(f'  {f.name:<40} {f.stat().st_size/1024**2:.1f} MB')

Reranker FT başlıyor | device: cuda


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

...

  step 100/274  avg_loss: 0.2473


Epoch 3/3:  73%|███████▎  | 200/274 [01:18<00:28,  2.60it/s]

  step 200/274  avg_loss: 0.2563


Epoch 3/3: 100%|██████████| 274/274 [01:47<00:00,  2.55it/s]


Epoch 3 bitti | avg_loss: 0.2543

✅ Reranker kaydedildi: /kaggle/working/reranker_ft
  config.json                              0.0 MB
  model.safetensors                        127.3 MB
  special_tokens_map.json                  0.0 MB
  tokenizer.json                           0.7 MB
  tokenizer_config.json                    0.0 MB
  vocab.txt                                0.2 MB


## Bölüm 2 — Qwen2.5-7B QLoRA SFT

In [6]:
llm_path = Path(CFG.DRIVE_DIR) / CFG.LLM_SFT_FILE
llm_raw  = []
with open(llm_path, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            llm_raw.append(json.loads(line))

print(f'LLM ham veri: {len(llm_raw)} satır')
print(f'Örnek roller: {[m["role"] for m in llm_raw[0].get("messages", [])]}')

random.seed(CFG.RANDOM_SEED)
sft_pool = llm_raw if len(llm_raw) <= CFG.QLORA_MAX_TRAIN else random.sample(llm_raw, CFG.QLORA_MAX_TRAIN)
print(f'Eğitim seti: {len(sft_pool)} örnek  (toplam: {len(llm_raw)})')

LLM ham veri: 13504 satır
Örnek roller: ['system', 'user', 'assistant']
Eğitim seti: 3000 örnek  (toplam: 13504)


In [7]:
from transformers import (
    AutoTokenizer as HFTok,
    AutoModelForCausalLM as HFLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
import trl
from trl import SFTTrainer
try:
    from trl import SFTConfig
    _HAS_SFTCONFIG = True
except ImportError:
    _HAS_SFTCONFIG = False
from datasets import Dataset as HFDataset

FT_LLM_PATH = Path(CFG.QLORA_OUT)

# Yarım kalmış adapter temizle
if FT_LLM_PATH.exists() and not (FT_LLM_PATH / 'adapter_config.json').exists():
    print('Yarım kalmış adapter temizleniyor...')
    shutil.rmtree(FT_LLM_PATH)

if FT_LLM_PATH.exists() and (FT_LLM_PATH / 'adapter_config.json').exists():
    print(f'✅ QLoRA zaten eğitilmiş: {FT_LLM_PATH}')
else:
    print('QLoRA SFT başlıyor...')
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print(f'Kullanılabilir VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

    llm_tok = HFTok.from_pretrained(CFG.LLM_MODEL, trust_remote_code=True)
    if llm_tok.pad_token is None:
        llm_tok.pad_token = llm_tok.eos_token
    llm_tok.padding_side = 'right'

    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
    )

    max_memory = None
    if torch.cuda.is_available() and torch.cuda.device_count() >= 2:
        max_memory = {0: '13GiB', 1: '13GiB', 'cpu': '30GiB'}

    llm_base = HFLM.from_pretrained(
        CFG.LLM_MODEL,
        quantization_config=bnb_cfg,
        device_map='auto',
        max_memory=max_memory,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        torch_dtype=torch.float16,
    )
    llm_base = prepare_model_for_kbit_training(llm_base)
    llm_base.config.use_cache = False
    print(f'Base model yüklendi | VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=CFG.QLORA_R,
        lora_alpha=CFG.QLORA_ALPHA,
        lora_dropout=CFG.QLORA_DROPOUT,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        bias='none',
    )
    llm_peft = get_peft_model(llm_base, lora_cfg)

    for p in llm_peft.parameters():
        if p.requires_grad:
            p.data = p.data.float()

    llm_peft.config.use_cache = False
    llm_peft.print_trainable_parameters()

    # messages → chat template formatına çevir
    def format_sample(item):
        msgs = item.get('messages', [])
        text = llm_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        return {'text': text}

    hf_dataset = HFDataset.from_list([format_sample(d) for d in sft_pool])
    print(f'SFT örnek: {len(hf_dataset)}')

    training_args = TrainingArguments(
        output_dir='/kaggle/working/qlora_ckpt',
        num_train_epochs=CFG.QLORA_EPOCHS,
        per_device_train_batch_size=CFG.QLORA_BATCH,
        gradient_accumulation_steps=CFG.QLORA_GRAD_ACC,
        learning_rate=CFG.QLORA_LR,
        warmup_ratio=0.05,
        lr_scheduler_type='cosine',
        fp16=False,
        bf16=False,
        max_grad_norm=1.0,       # FİX: 0.0 → 1.0 (gradient clipping açık)
        logging_steps=50,
        save_strategy='no',
        report_to='none',
        optim='paged_adamw_8bit',
        dataloader_num_workers=0,
        remove_unused_columns=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
    )

    trl_version = tuple(int(x) for x in trl.__version__.split('.')[:2])
    print(f'TRL versiyonu: {trl.__version__}')

    if _HAS_SFTCONFIG and trl_version >= (0, 9):
        # TRL >= 0.9: max_seq_length SFTConfig'e taşındı
        # TRL 0.12+'da max_length oldu — her ikisini de dene
        _sft_kwargs = dict(
            output_dir='/kaggle/working/qlora_ckpt',
            num_train_epochs=CFG.QLORA_EPOCHS,
            per_device_train_batch_size=CFG.QLORA_BATCH,
            gradient_accumulation_steps=CFG.QLORA_GRAD_ACC,
            learning_rate=CFG.QLORA_LR,
            warmup_ratio=0.05,
            lr_scheduler_type='cosine',
            fp16=False,
            bf16=False,
            max_grad_norm=1.0,
            logging_steps=50,
            save_strategy='no',
            report_to='none',
            optim='paged_adamw_8bit',
            dataloader_num_workers=0,
            remove_unused_columns=False,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={'use_reentrant': False},
            dataset_text_field='text',
            packing=False,
        )
        try:
            sft_cfg = SFTConfig(max_seq_length=CFG.QLORA_MAX_SEQ, **_sft_kwargs)
            print('SFTConfig: max_seq_length kullanıldı')
        except TypeError:
            try:
                sft_cfg = SFTConfig(max_length=CFG.QLORA_MAX_SEQ, **_sft_kwargs)
                print('SFTConfig: max_length kullanıldı')
            except TypeError:
                sft_cfg = SFTConfig(**_sft_kwargs)
                llm_tok.model_max_length = CFG.QLORA_MAX_SEQ
                print(f'SFTConfig: tokenizer.model_max_length={CFG.QLORA_MAX_SEQ} ile truncation')
        trainer = SFTTrainer(
            model=llm_peft,
            processing_class=llm_tok,
            train_dataset=hf_dataset,
            args=sft_cfg,
        )
    else:
        # TRL < 0.9: max_seq_length SFTTrainer parametresi
        trainer = SFTTrainer(
            model=llm_peft,
            processing_class=llm_tok,
            train_dataset=hf_dataset,
            args=training_args,
            max_seq_length=CFG.QLORA_MAX_SEQ,
            dataset_text_field='text',
        )
    print(f'Trainer hazır — eğitim başlıyor...')
    trainer.train()

    FT_LLM_PATH.mkdir(parents=True, exist_ok=True)
    llm_peft.save_pretrained(str(FT_LLM_PATH))
    llm_tok.save_pretrained(str(FT_LLM_PATH))

    del llm_peft, llm_base, trainer, hf_dataset
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    print(f'\n✅ QLoRA adapter kaydedildi: {FT_LLM_PATH}')
    for f in sorted(FT_LLM_PATH.rglob('*')):
        if f.is_file():
            print(f'  {f.name:<40} {f.stat().st_size/1024**2:.1f} MB')

print('QLoRA tamamlandı.')

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Yarım kalmış adapter temizleniyor...
QLoRA SFT başlıyor...
Kullanılabilir VRAM: 14.6 GB


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Trainer hazır — eğitim başlıyor...


Step,Training Loss
50,0.977000



✅ QLoRA adapter kaydedildi: /kaggle/working/qwen_qlora_ft
  README.md                                0.0 MB
  adapter_config.json                      0.0 MB
  adapter_model.safetensors                38.5 MB
  added_tokens.json                        0.0 MB
  chat_template.jinja                      0.0 MB
  merges.txt                               1.6 MB
  special_tokens_map.json                  0.0 MB
  tokenizer.json                           10.9 MB
  tokenizer_config.json                    0.0 MB
  vocab.json                               2.6 MB
QLoRA tamamlandı.


## Bölüm 3 — Kaggle Dataset'e Upload

**Önemli:** Her iki model tek seferde upload ediliyor.
S6/S7/S8 notebookları şu yapıyı bekliyor:
```
llegal-models-tr-finetuned/
  bgem3/          ← P1 v2'den geliyor
  reranker_ft/    ← bu notebook
  qwen_qlora_ft/  ← bu notebook
```

In [8]:
# Her iki modeli tek bir staging klasörüne topla
STAGING = Path('/kaggle/working/legal-models-tr-finetuned_staging')
STAGING.mkdir(parents=True, exist_ok=True)

# reranker_ft kopyala
rk_staging = STAGING / 'reranker_ft'
if rk_staging.exists():
    shutil.rmtree(rk_staging)
shutil.copytree(CFG.RERANKER_OUT, rk_staging)
print(f'✅ reranker_ft kopyalandı → {rk_staging}')

# qwen_qlora_ft kopyala
qlora_staging = STAGING / 'qwen_qlora_ft'
if qlora_staging.exists():
    shutil.rmtree(qlora_staging)
shutil.copytree(CFG.QLORA_OUT, qlora_staging)
print(f'✅ qwen_qlora_ft kopyalandı → {qlora_staging}')

# Dataset metadata
meta = {
    'title'   : CFG.KAGGLE_DATASET_ID.split('/')[1],
    'id'      : CFG.KAGGLE_DATASET_ID,
    'licenses': [{'name': 'CC0-1.0'}]
}
with open(STAGING / 'dataset-metadata.json', 'w') as f:
    json.dump(meta, f)

total_mb = sum(f.stat().st_size for f in STAGING.rglob('*') if f.is_file()) / 1024**2
print(f'\nToplam upload boyutu: {total_mb:.0f} MB')

# Mevcut dataset'e yeni versiyon ekle
ret = os.system(f'kaggle datasets version -p "{STAGING}" -m "add reranker_ft + qwen_qlora_ft" --dir-mode zip')

if ret != 0:
    # İlk kez oluşturuluyorsa
    ret = os.system(f'kaggle datasets create -p "{STAGING}" --dir-mode zip')

if ret == 0:
    print('\n✅ Dataset başarıyla yüklendi!')
else:
    print('\n❌ Otomatik upload başarısız. Manuel komut:')
    print(f'   !kaggle datasets version -p {STAGING} -m "add models" --dir-mode zip')

✅ reranker_ft kopyalandı → /kaggle/working/legal-models-tr-finetuned_staging/reranker_ft
✅ qwen_qlora_ft kopyalandı → /kaggle/working/legal-models-tr-finetuned_staging/qwen_qlora_ft

Toplam upload boyutu: 182 MB
Starting upload for file reranker_ft.zip


100%|██████████| 117M/117M [00:01<00:00, 97.0MB/s] 


Upload successful: reranker_ft.zip (117MB)
Starting upload for file qwen_qlora_ft.zip


100%|██████████| 34.1M/34.1M [00:00<00:00, 54.6MB/s]


Upload successful: qwen_qlora_ft.zip (34MB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/ardayildiz29/legal-models-tr-finetuned

✅ Dataset başarıyla yüklendi!


In [9]:
print('=' * 60)
print('P2 v2 EĞİTİM TAMAMLANDI')
print('=' * 60)
print()
print('Yapılan düzeltmeler:')
print('  ✅ Reranker: kısa query filtresi (< 3 kelime atlandı)')
print('  ✅ Reranker: pos_weight ile class imbalance düzeltmesi')
print(f'  ✅ QLoRA: max_seq_length={CFG.QLORA_MAX_SEQ} eklendi')
print('  ✅ QLoRA: max_grad_norm=1.0 (gradient clipping açık)')
print(f'  ✅ QLoRA: MAX_TRAIN={CFG.QLORA_MAX_TRAIN} (2000 yerine)')
print('  ✅ Upload: tek staging klasöründen (üst yazma sorunu giderildi)')
print()
print('Diğer notebooklarda model yolları (değişiklik gerekmez):')
print('  MODEL_DIR      = "/kaggle/input/datasets/ardayildiz29/legal-models-tr-finetuned')
print('  FT_EMBED_PATH  = Path(MODEL_DIR) / "bgem3"')
print('  FT_RERANKER    = Path(MODEL_DIR) / "reranker_ft"')
print('  FT_LLM         = Path(MODEL_DIR) / "qwen_qlora_ft"')

P2 v2 EĞİTİM TAMAMLANDI

Yapılan düzeltmeler:
  ✅ Reranker: kısa query filtresi (< 3 kelime atlandı)
  ✅ Reranker: pos_weight ile class imbalance düzeltmesi
  ✅ QLoRA: max_seq_length=512 eklendi
  ✅ QLoRA: max_grad_norm=1.0 (gradient clipping açık)
  ✅ QLoRA: MAX_TRAIN=3000 (2000 yerine)
  ✅ Upload: tek staging klasöründen (üst yazma sorunu giderildi)

Diğer notebooklarda model yolları (değişiklik gerekmez):
  MODEL_DIR      = "/kaggle/input/datasets/ardayildiz29/legal-models-tr-finetuned
  FT_EMBED_PATH  = Path(MODEL_DIR) / "bgem3"
  FT_RERANKER    = Path(MODEL_DIR) / "reranker_ft"
  FT_LLM         = Path(MODEL_DIR) / "qwen_qlora_ft"
